# Darija Medical Intent Classifier — Fine-tuning XLM-RoBERTa
**Dissertation: Darija Medical Assistant**

This notebook fine-tunes XLM-RoBERTa (a multilingual model) on a Darija+English medical symptom dataset to classify user intent into 10 categories.

**Runtime:** Make sure you are using a GPU runtime: Runtime → Change runtime type → T4 GPU

**Categories:**
- headache, fever, stomach, cough_cold, pain, sleep, fatigue, skin, mental, chronic

In [ ]:
# Step 1: Install dependencies
!pip install transformers datasets scikit-learn torch -q

In [ ]:
# Step 2: Upload your dataset
# Run this cell, then upload darija_medical_dataset.json when prompted
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Step 3: Load and prepare dataset
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

with open('darija_medical_dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f'Total examples: {len(df)}')
print(df['label'].value_counts())

# Encode labels to numbers
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
label_names = le.classes_.tolist()
num_labels = len(label_names)
print(f'\nClasses ({num_labels}):', label_names)

# Train/test split (80/20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])
print(f'\nTrain: {len(train_df)} | Test: {len(test_df)}')

In [ ]:
# Step 4: Tokenise with XLM-RoBERTa
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=64)

train_ds = Dataset.from_pandas(train_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}))
test_ds  = Dataset.from_pandas(test_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}))

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print('Tokenisation complete.')

In [ ]:
# Step 5: Load model and train
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='./darija-intent-model',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('Starting training...')
trainer.train()

In [ ]:
# Step 6: Evaluate and show results
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
true  = predictions.label_ids

print('=== TEST SET RESULTS ===')
print(f'Accuracy: {accuracy_score(true, preds)*100:.1f}%')
print()
print(classification_report(true, preds, target_names=label_names))

In [ ]:
# Step 7: Test with your own inputs
import torch

def predict_intent(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred_id = outputs.logits.argmax(-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1)[0][pred_id].item()
    return label_names[pred_id], round(confidence * 100, 1)

# Test examples
test_inputs = [
    'rasi kaydrni bzzaf',
    'عندي سخانة وبرودة',
    'I have stomach pain',
    'ما كنعسش',
    'my back is killing me',
    'عندي السكري',
    'كنحس بالقلق',
    'I have a rash on my arm',
]

print('=== LIVE PREDICTIONS ===')
for t in test_inputs:
    intent, conf = predict_intent(t)
    print(f'  "{t}" → {intent} ({conf}%)')

In [ ]:
# Step 8: Save label mapping and download model
import json, shutil

# Save label names so we can decode predictions later
with open('./darija-intent-model/label_names.json', 'w') as f:
    json.dump(label_names, f)

# Save tokenizer alongside model
tokenizer.save_pretrained('./darija-intent-model')

# Zip and download
shutil.make_archive('darija_intent_model', 'zip', './darija-intent-model')
files.download('darija_intent_model.zip')
print('Model saved and downloading...')